# Porto taxi dataset

In [14]:
from datetime import datetime

import numpy as np
import pandas as pd

raw_dataset = pd.read_csv(
    "assets/taxi_porto.csv",
    usecols=["TIMESTAMP", "POLYLINE"],
)

index = pd.date_range(
    start=datetime.fromtimestamp(min(raw_dataset["TIMESTAMP"])), 
    end=datetime.fromtimestamp(max(raw_dataset["TIMESTAMP"])), 
    freq="15min"
)

In [2]:
raw_dataset = raw_dataset.dropna()

In [3]:
raw_dataset.head()

,TIMESTAMP,POLYLINE
0,1372636858,"[[-8.618643,41.141412],[-8.618499,41.141376],[..."
1,1372637303,"[[-8.639847,41.159826],[-8.640351,41.159871],[..."
2,1372636951,"[[-8.612964,41.140359],[-8.613378,41.14035],[-..."
3,1372636854,"[[-8.574678,41.151951],[-8.574705,41.151942],[..."
4,1372637091,"[[-8.645994,41.18049],[-8.645949,41.180517],[-..."


In [15]:
from math import floor

LON_MIN = -8.6338
LON_MAX = -8.5862
LAT_MIN = 41.1369
LAT_MAX = 41.1690
BBox = (LON_MIN, LON_MAX, LAT_MIN, LAT_MAX)

N_ROWS = 20
N_COLS = 20
N_CELLS = N_ROWS * N_COLS

LAT_DIFF = LAT_MAX - LAT_MIN
LON_DIFF = LON_MAX - LON_MIN

CELL_LON = LON_DIFF / N_COLS
CELL_LAT = LAT_DIFF / N_ROWS

def get_cell(lat: float, lon: float) -> int:
    lon_delta = lon - LON_MIN
    col = floor(lon_delta / CELL_LON)
    
    lat_delta = lat - LAT_MIN
    row = floor(lat_delta / CELL_LAT)
    
    return row*N_COLS + col

In [5]:
gps = raw_dataset["POLYLINE"].apply(eval).dropna()
gps = gps.rename("gps")

In [6]:
gps_dataset = pd.DataFrame(data={"timestamp": raw_dataset["TIMESTAMP"], "gps": gps})

In [7]:
df = pd.DataFrame(
    index=index,
    data=np.zeros((len(index), N_CELLS)),
    columns=[f"cell_{i}" for i in range(N_CELLS)]
)

In [8]:
base_ts = min(raw_dataset["TIMESTAMP"])

for i, g in gps_dataset.iterrows():
    first_ts = g["timestamp"]
    trajectory = g["gps"]
    if isinstance(trajectory, list) and len(trajectory) > 0:
        for j, (lon, lat) in enumerate(trajectory):
            if LON_MIN < lon < LON_MAX and LAT_MIN < lat < LAT_MAX:
                cell = get_cell(lat=lat, lon=lon)
                ts = first_ts + j*15
                
                row = floor((ts - base_ts) / (60 * 15))
                
                df.iloc[row][cell] += 1

/tmp/ipykernel_62535/1204293363.py:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df.iloc[row][cell] += 1
/tmp/ipykernel_62535/1204293363.py:14: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

S

In [9]:
df.head()

,cell_0,cell_1,cell_2,cell_3,cell_4,cell_5,cell_6,cell_7,cell_8,cell_9,...,cell_390,cell_391,cell_392,cell_393,cell_394,cell_395,cell_396,cell_397,cell_398,cell_399
2013-07-01 02:00:53,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0
2013-07-01 02:15:53,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.0,3.0,...,0.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,6.0
2013-07-01 02:30:53,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,2.0,0.0,0.0,0.0,0.0,0.0,6.0,1.0,4.0
2013-07-01 02:45:53,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,4.0,1.0,2.0
2013-07-01 03:00:53,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,2.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [10]:
df.to_csv("assets/porto_taxi_cells.csv")

### Assignment to nodes

In [16]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

n_nodes = 20
k=19
n_simulations = 10
network_path = Path(f"data/networks/porto_{n_nodes}n_{k}k")
output_path = Path(f"data/datasets/porto_{n_nodes}n_{k}k")

In [17]:
from category_encoders import BinaryEncoder
import pandas as pd

df = pd.read_csv("assets/porto_taxi_cells.csv", index_col=0, parse_dates=True)

def encode_time(df: pd.DataFrame) -> pd.DataFrame:
    df["hour"] = df.index.hour
    df["weekday"] = df.index.weekday
    return BinaryEncoder(cols=["hour", "weekday"]).fit_transform(df)

In [18]:
for n in range(n_simulations):
    dataset_path = output_path / str(n)
    dataset_path.mkdir(exist_ok=True, parents=True)
    
    towers = pd.read_csv(network_path / str(n) / "towers.csv")

    tower_cells = [
        [] for i in range(n_nodes)
    ]
    
    for i in range(N_ROWS):
        for j in range(N_COLS):
            cell = i*N_ROWS + j
            
            lat_begin = LAT_MIN + i*CELL_LAT
            lon_begin = LON_MIN + j*CELL_LON
            
            lat_center = lat_begin + CELL_LAT / 2
            lon_center = lon_begin + CELL_LON / 2
            
            min_dist = None
            nearest_tower = None
            for n_tower, tower in towers.iterrows():
                dist = (lat_center - tower["lat"])**2 + (lon_center - tower["lon"])**2
                if min_dist is None or min_dist > dist:
                    min_dist = dist
                    nearest_tower = n_tower
                    
            tower_cells[nearest_tower].append(cell)
            
    tower_calls = pd.DataFrame(
        np.zeros((len(index), n_nodes)), 
        columns=[f"tower_{i}" for i in range(n_nodes)], 
        index=index
    )
    
    for i, cells in enumerate(tower_cells):
        ind = f"tower_{i}"
        for cell in cells:
            tower_calls[ind] += df[f"cell_{cell}"]
            
        # tower_calls[ind] += 1
        # tmp_ind = tower_calls[ind][1:].index
        # tower_calls[ind] = pd.Series(
        #     tower_calls[ind][1:].to_numpy() / tower_calls[ind][:-1].to_numpy(),
        #     index=tmp_ind,
        # )
    
                        
    quantiles = tower_calls.quantile(0.8)
    tower_calls = tower_calls.clip(0, quantiles, axis=1)

    # max_value = tower_calls.max().max()
    # if max_value > 0:
    #     tower_calls = tower_calls / max_value
    # else:
    #     print("Max value is 0!!")
    
    node_datasets = [
        encode_time(pd.DataFrame({"requests": tower_calls[col]}))
        for col in tower_calls.columns
    ]
    
    for i, ds in enumerate(node_datasets):
        ds = ds.dropna()
        ds.to_parquet(dataset_path / f"node_{i}_notscaled.parquet")
    

/home/atundo/Work/gl-forecasting-edge/.venv/lib/python3.11/site-packages/sklearn/base.py:411: FutureWarning: The `_get_tags` method is deprecated in 1.6 and will be removed in 1.7. Please implement the `__sklearn_tags__` method.
  warnings.warn(
/home/atundo/Work/gl-forecasting-edge/.venv/lib/python3.11/site-packages/sklearn/base.py:411: FutureWarning: The `_get_tags` method is deprecated in 1.6 and will be removed in 1.7. Please implement the `__sklearn_tags__` method.
  warnings.warn(
/home/atundo/Work/gl-forecasting-edge/.venv/lib/python3.11/site-packages/sklearn/base.py:411: FutureWarning: The `_get_tags` method is deprecated in 1.6 and will be removed in 1.7. Please implement the `__sklearn_tags__` method.
  warnings.warn(
/home/atundo/Work/gl-forecasting-edge/.venv/lib/python3.11/site-packages/sklearn/base.py:411: FutureWarning: The `_get_tags` method is deprecated in 1.6 and will be removed in 1.7. Please implement the `__sklearn_tags__` method.
  warnings.warn(
/home/atundo/Wor

In [20]:
from utils.data import prepare_dataset_for_training

ds_name = "4in_notscaled"

for n in range(n_simulations):
    towers_datasets = [
        pd.read_parquet(output_path / str(n) / f"node_{i}_notscaled.parquet") for i in range(n_nodes)
    ]
    
    small_ds_nodes = list(range(10))
    
    start_i = round(len(towers_datasets[0]) * 0.8)
    start_ind = index[start_i]
    
    towers_datasets = [
        ts[start_ind:] 
        if i in small_ds_nodes
        else ts
        for i, ts in enumerate(towers_datasets)
    ]
    
    # scale
    # max_value = max(*[ts['requests'].max() for ts in towers_datasets])
    # for i in range(len(towers_datasets)):
    #     towers_datasets[i]["requests"] /= max_value
    # 
    # folder = output_path / str(n) / ds_name
    # folder.mkdir(parents=True, exist_ok=True)
    # with (folder / "scaling_factor.txt").open("w") as f:
    #     f.write(str(max_value))
    
    prepare_dataset_for_training(
        towers_datasets=towers_datasets,
        output_folder=output_path / str(n) / ds_name,
        input_timesteps=4,
        output_timesteps=1,
        n_functions=1,
        n_auxiliary_features=8,
    )

Saved!

Saved!

Saved!

Saved!

Saved!

Saved!

Saved!

Saved!

Saved!

Saved!

